# Step 04.2 — Parameter、Bias 与 `nn.Module`

4.1 已经把 `nn.Linear` 拆回了矩阵乘法。现在继续回答：**为什么一个普通矩阵 `W` 放进神经网络后会变成“参数”？bias 又是什么？MLX 怎么管理成百上千个这样的数组？**

本节分 4 部分：

1. Parameter：模型真正“学”的是什么；
2. Bias：为什么除了 `W` 还常常要加一个 `b`；
3. `nn.Module`：为什么层和模型要写成对象；
4. Parameter tree：MLX 怎么自动找到、组织和统计参数。

## 1. Parameter：模型真正会被训练修改的数字

先区分三类东西：

```text
输入 data          x           每个样本会变
模型结构设置       2 -> 3      由我们决定
模型 parameter     weight      训练过程中会被更新
```

例如：

```python
layer = nn.Linear(2, 3, bias=False)
```

`2` 和 `3` 决定这一层的结构；它们不是模型自己学出来的。

真正需要学习的是 `layer.weight` 里面的 6 个浮点数：

```text
(3,2) -> 3 × 2 = 6 个 parameter
```

现在它们只是随机初始化值。后面训练时，loss 会告诉我们预测有多差，gradient 会告诉每个 parameter 应该往哪个方向改。

In [ ]:
import mlx.core as mx
import mlx.nn as nn

layer_no_bias = nn.Linear(2, 3, bias=False)

print("weight shape:", layer_no_bias.weight.shape)
print("weight size :", layer_no_bias.weight.size)
print("parameters  :", layer_no_bias.parameters())

`weight.size = 6`，表示这一层目前有 6 个可以被模型管理的权重数值。

注意：**parameter 不等于“已经学会的知识”**。初始化时它只是随机数字；经过训练后，才逐渐成为对任务有用的数值。

## 2. Bias：从 `xWᵀ` 到 `xWᵀ + b`

没有 bias 时：

```text
y = xWᵀ
```

有 bias 时：

```text
y = xWᵀ + b
```

最简单的一维情况：

```text
y = 2x
```

当 `x=0` 时一定有 `y=0`。这条直线必须经过原点。

如果加入：

```text
y = 2x + 3
```

当 `x=0` 时，`y=3`。bias 让整个变换可以整体平移，而不必被固定在原点。

因此严格数学上，带 bias 的 `Wx+b` 是 **affine transformation（仿射变换）**，但神经网络库仍习惯把这一层叫 `Linear`。

In [ ]:
layer = nn.Linear(2, 3, bias=True)

print("weight shape:", layer.weight.shape)
print("bias shape  :", layer.bias.shape)
print("weight count:", layer.weight.size)
print("bias count  :", layer.bias.size)

`Linear(2,3)` 有 3 个输出，所以 bias 也需要 3 个数：

```text
weight: (3,2) -> 6 个
bias  : (3,)  -> 3 个
总参数          9 个
```

如果输入是一整个 batch：

```text
X @ W.T : (B,3)
bias    :    (3,)
```

你在 Step 3 学过 broadcasting，所以 `(3,)` 会自动加到 batch 的每一行。

In [ ]:
X = mx.array([
    [2.0, -1.0],
    [3.0,  4.0],
])

from_layer = layer(X)
manual = X @ layer.weight.T + layer.bias

mx.eval(from_layer, manual)

print("layer(X):")
print(from_layer)
print("\nmanual:")
print(manual)
print("\nsame:", bool(mx.allclose(from_layer, manual)))

如果输出 `same: True`，就说明你已经把带 bias 的 `nn.Linear` 也拆掉了黑盒：

```python
layer(X)
```

本质就是：

```python
X @ layer.weight.T + layer.bias
```

## 3. `nn.Module`：为什么模型要写成类？

到目前为止，一层 Linear 已经需要保存：

```text
weight
bias
forward rule
```

以后 MiniGPT 会有几十个乃至更多这样的层。如果只用散落的 Python 变量，我们就得自己手动维护所有参数。

`nn.Module` 的主要作用之一，就是提供一个统一的**神经网络容器**：一个 Module 可以保存自己的数组参数，也可以包含其他 Module。

我们先写一个极小的自定义模型。

In [ ]:
class TinyProjector(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(2, 3, bias=True)

    def __call__(self, x):
        return self.proj(x)


model = TinyProjector()
print(model)

逐行看：

```python
class TinyProjector(nn.Module):
```

表示 `TinyProjector` 继承 MLX 的 `nn.Module`，因此获得 Module 的参数管理能力。

```python
super().__init__()
```

调用父类 `nn.Module` 的初始化逻辑。

```python
self.proj = nn.Linear(...)
```

把另一个 Module（Linear）挂到当前模型里面，成为 child module。

```python
def __call__(self, x):
    return self.proj(x)
```

定义模型收到输入时的 forward path。

因此：

```python
model(X)
```

会调用 `TinyProjector.__call__`，而它内部又调用 `self.proj(X)`。

In [ ]:
out = model(X)
mx.eval(out)

print("input shape :", X.shape)
print("output shape:", out.shape)

## 4. Parameter tree：MLX 怎么找到所有参数？

在 MLX 的 `nn.Module` 中，公开的 `mlx.core.array` 成员会被 Module 递归收集为参数；子 Module 里的参数也会一起被找到。

所以虽然我们的 `TinyProjector` 自己只写了：

```python
self.proj = nn.Linear(...)
```

但 `model.parameters()` 可以继续进入 `proj`，找到它的 `weight` 和 `bias`。


In [ ]:
params = model.parameters()

print("parameter tree:")
print(params)

print("\nweight shape:", params["proj"]["weight"].shape)
print("bias shape  :", params["proj"]["bias"].shape)

它不是把所有东西压成一个大数组，而是保留模型结构，形成类似：

```text
{
  "proj": {
      "weight": ...,
      "bias": ...
  }
}
```

这就是 parameter tree（参数树）的直觉。以后 MiniGPT 有 `blocks → attention → c_attn → weight` 等多层嵌套时，MLX 仍然可以递归管理。

In [ ]:
from mlx.utils import tree_flatten

flat_params = tree_flatten(model.parameters())

for name, value in flat_params:
    print(name, value.shape, "count =", value.size)

num_params = sum(value.size for _, value in flat_params)
print("\ntotal parameters:", num_params)

这里应该得到总参数数：

```text
weight: 3×2 = 6
bias  : 3   = 3
total       = 9
```

这就是以后我们说“这个模型有多少参数”的最基本计算方式。

### 一个 MLX 特别要注意的点

在 `nn.Module` 里，公开的 `mx.array` 成员默认会进入参数体系，而且默认可训练，除非被 freeze。

因此以后像 causal mask 这种**固定常量**，不能随便当作普通公开参数挂进去。我们到 Attention 时会专门处理这个问题。

### 4.2 最终结论

- Parameter：模型训练时会被优化器修改的数值；
- `weight` 决定输入特征怎样组合成输出特征；
- `bias` 给每个输出增加独立平移量；
- `Linear(2,3,bias=True)` 一共有 `3×2 + 3 = 9` 个参数；
- `nn.Module` 是组织参数和子层的容器；
- `model.parameters()` 会递归形成 parameter tree；
- 后面训练不需要我们手工逐个寻找 `weight`，MLX 可以从 Module 的参数树统一取得它们。

下一节 4.3：为什么只堆 Linear 仍然不够，以及 ReLU / GELU 为什么会出现。